# Sod Shock Tube (3D)

The same Riemann problem as `sod_1d.ipynb`, extruded twice: a periodic slab
in both $y$ and $z$. `sod_2d.ipynb` is the same notebook one dimension down,
and most of what is said there applies here unchanged.

Side | $p$ | $\rho$ | $v$
---|---|---|---
Left | 1.0 | 1.0 | 0.0
Right | 0.1795 | 0.25 | 0.0

* **$x$** is arranged exactly as in 1D: periodic on $[-1, 1]$, the dense
  state occupying the middle half $|x| \le 0.5$ and the light state the two
  outer quarters. Nothing is reflected explicitly -- the mirror symmetry of
  that arrangement makes $x = 0$ and $x = \pm 1$ behave as reflecting walls
  until a wave reaches them. The analytic solution describes the window
  $x \in [0, 1]$, which is what the profile panels draw.
* **$y$ and $z$** are plain periodic directions, both of the same width,
  measured in particle spacings rather than as a length (see the parameters
  cell).

**Equal mass is not exactly reachable here.** The light state has to be
sampled $(\rho_l/\rho_r)^{1/3} = 4^{1/3} \approx 1.587$ times coarser in
every direction to give both states the same particle mass -- and that is
irrational, so no pair of commensurate periodic lattices matches exactly, the
way 1D's `samplingRatio=4` and 2D's $\sqrt{4} = 2$ do. The sampler picks two
integers instead (transverse count from the isotropic ideal, $x$ count for the
mass match, then retries the neighbouring transverse counts) and lands within
about 1%, against the 75% that equal *spacing* would leave. The sampling cell
below prints what it settled on and what it would have cost to ask for
something else.

**How to read the profile panels.** Every particle is scattered against its
own $x$, with no averaging over the cross-section -- the domain is periodic in
$y$ and $z$ and the solution does not depend on either, so all of them are
directly comparable to the same 1D reference curve. The *vertical spread* at
a given $x$ is the symmetry breaking, now measured over a two-dimensional
cross-section rather than a line. The last cell cuts a thin slice through the
slab and draws the density in it, which is where that spread becomes a
picture.

**A note on trusting 3D output.** This case was the first thing in the repo
to run any 3D physics, and it immediately found a real bug: `warpSPHCore`'s
B7 kernel -- the default here -- had a 3D normalisation constant 16x too
small, so every density came back at $1/16$ of the mass it was built from,
while the velocities and the wave *positions* still looked entirely right.
Fixed, and checked by summing every kernel over a uniform lattice of known
density in all three dimensions. Worth knowing which way that failure looks:
a uniformly wrong density with a plausible wave structure is a normalisation
problem, not a physics one.

**Cost.** ~20k particles at the default `nx=40`, since 3D pays the transverse
count twice; the resolution along the tube is correspondingly lower than the
2D notebook's. `nx` scales it and the slab follows automatically, being
measured in spacings.

Like `sod_1d.ipynb`, this notebook is meant to be **edited while it runs**:
the step loop stays unrolled in a cell rather than hidden inside
`warpSPH.runner.run()`. Plotting calls `plotSod`/`plotSod_` directly rather
than going through `sod3dCase.setupPlot` -- see `sod_1d.ipynb`'s intro for why
that path does not live-update inside a Jupyter cell.

Precision note: switching between single and double precision is controlled
in the import cell below, and requires a kernel restart to take effect.

![](outputs/01-Sod_Shock_Tube_3D.gif)

In [ ]:
%matplotlib widget
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float32', verbose=True)

from warpSPH import *
from warpSPH.cases.sod import states
from warpSPH.cases.sodND import sod3dCase
from warpSPH.caseUtils import plotSod, plotSod_, sodSampling, sodSamplingReport
from warpSPH.runner import CaseSpec, buildContext, encodeFrames
from warpSPH.runner.display import figureOf, visualizeWithFallback
from warpSPH.io import createOutFile, prepExport, writeInitialData, writeFrame

import dataclasses
import os
import torch
from tqdm.autonotebook import tqdm

In [ ]:
# Every knob you'd otherwise reach for as a `--flag` on `sod_3d.py`, made
# explicit and editable here. `sod3dCase.defaults`/`.params` are the same
# values the CLI script starts from -- anything not overridden below just keeps
# its case default.
spec = CaseSpec(caseName=sod3dCase.name, scheme=sod3dCase.scheme,
                params=dict(sod3dCase.params)).merged(**sod3dCase.defaults)

spec = spec.merged(
    # --- discretisation --------------------------------------------------
    nx=40,                     # dense-side particles across its own half of the domain
    dim=3,
    L=2.0,

    # --- time stepping -----------------------------------------------------
    tLimit=0.15,
    dt=1e-3, adaptiveDt=True, cflFactor=0.3,

    # --- scheme --------------------------------------------------------
    kernel='B7',

    # --- output ------------------------------------------------------------
    plot=True, show=True, plotInterval=10,
    store=True,
    storeMode='trajectory',    # one growing trajectory.h5 (see sod_resume.ipynb)
    exportInterval=0.005,      # simulated-time interval between stored frames

    params=dict(
        gamma=5 / 3,
        left_rho=1.0, left_pressure=1.0, left_velocity=0.0,
        right_rho=0.25, right_pressure=0.1795, right_velocity=0.0,
        # The slab's width, in dense-side particle spacings. Measured that way
        # rather than as a length because the constraint it has to satisfy is a
        # multiple of the spacing: a slab narrower than twice the support radius
        # lets a particle interact with its own periodic image, silently. Fixing
        # it in spacings keeps that true at every `nx` -- the sampler checks and
        # raises if it ever is not -- and makes the transverse particle count
        # independent of resolution, so the total grows linearly in `nx` rather
        # than quadratically.
        transverseSpacings=20,
        # Set False to sample both states on the same lattice instead, leaving
        # the dense side's particles 4x heavier. The sampling cell below prints
        # what that costs before you run anything.
        equalMass=True,
    ),
)
spec

In [ ]:
# Initial-condition generation: explicit, using the real case code
# (`sod3dCase.buildSystem` -> `buildSodND`), not re-derived here. The build
# reports the lattice it settled on, and replaces the domain's transverse
# bounds with the slab it snapped to.
ctx = buildContext(sod3dCase, spec)
sod3dCase.configureScheme(ctx)
system = sod3dCase.buildSystem(ctx)
runningState = system.initializeNewState()

left, right = states(ctx)
print(f'\ndomain {ctx.config.domain.min.tolist()} .. {ctx.config.domain.max.tolist()}')
print(f'{runningState.state.positions.shape[0]} particles, dt = {float(ctx.config.dt):.4g}')

## What the sampler picked, and what the alternative costs

`sodSampling` is pure arithmetic -- no particles, no kernels -- so the two
integers it has to choose for the light side can be inspected on their own,
at any resolution, without building anything. This is where the 3D case earns
its own sampler: the mismatch never quite reaches zero the way it does in 1D
and 2D, and which pair of integers is *least* wrong changes with `nx`.

In [ ]:
for equalMass in (True, False):
    sampling = sodSampling(spec.nx, spec.dim, spec.L, spec.param('transverseSpacings'),
                           left, right, equalMass=equalMass)
    print(f'equalMass={equalMass}:')
    print(sodSamplingReport(sampling))

print('\nacross resolutions (equalMass=True):')
for nx in (16, 25, 32, 40, 64):
    sampling = sodSampling(nx, spec.dim, spec.L, spec.param('transverseSpacings'), left, right)
    count = (sampling.dense[0] * sampling.dense[1] ** (spec.dim - 1)
             + sampling.light[0] * sampling.light[1] ** (spec.dim - 1))
    print(f'  nx={nx:>4d}: {count:>7d} particles, mass mismatch '
          f'{abs(sampling.massRatio - 1):>7.3%}, worst cell aspect {sampling.anisotropy:.4f}')

In [ ]:
# Export/plot setup via the same generic hooks `warpSPH.runner.run()` uses
# internally -- nothing here is re-derived, only called explicitly.
ctx.exportPath = prepExport(spec.caseName, ctx.config, ctx.schemeConfig, ctx.scheme,
                            ctx.exportFunction)
spec.save(os.path.join(ctx.exportPath, 'caseSpec.json'))
print(f'exporting to {ctx.exportPath}')

# `scatter=True` from the first frame on, unlike the 1D notebook: many
# particles share an x here, so a line through them would be meaningless.
fig = axis = None
if spec.plot:
    ctx.imagePath = os.path.join(ctx.exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    fig, axis = plotSod(runningState.state, ctx.config, ctx.schemeConfig, ctx.config.domain,
                        ctx.param('gamma'), left, right,
                        plotReference=True, plotLabels=False, scatter=True, t_=runningState.t)
    fig.savefig(os.path.join(ctx.imagePath, 'frame_00000.png'))

outFile = None
groups = None
if spec.store and spec.storeMode == 'trajectory':
    extraData = sod3dCase.extraData(ctx, runningState)
    outFile = createOutFile(ctx.exportPath)
    groups = writeInitialData(ctx.exportPath, outFile, ctx.scheme, ctx.config, ctx.schemeConfig,
                              spec, runningState, extraData=extraData,
                              extraFields=sod3dCase.extraFields)

In [ ]:
# The step loop, visible and editable. This is the same call
# `warpSPH.runner.runner._run` makes internally, unrolled here so a
# perturbation, an extra diagnostic, or a gradient step can be injected
# directly around it (`sod_backprop.ipynb` does the latter, in 1D).
dt = ctx.config.dt if isinstance(ctx.config.dt, float) else ctx.config.dt.cpu().item()
nSteps = int(spec.tLimit / dt)
storeSteps = max(1, int(spec.exportInterval / dt))

trajectory = []
for i in (tq := tqdm(range(nSteps), leave=True)):
    # <-- hook point -------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    # -----------------------------------------------------------------------

    tScalar = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = sod3dCase.diagnostics(ctx, runningState)
    trajectory.append(dict(row, step=i, t=tScalar))
    tq.set_description(f"t: {tScalar:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items()))

    if fig is not None and (i % spec.plotInterval == 0 or i == nSteps - 1):
        for ax in axis.flatten():
            ax.clear()
        plotSod_(fig, axis, runningState.state, ctx.config, ctx.schemeConfig, ctx.config.domain,
                 ctx.param('gamma'), left, right,
                 plotReference=True, plotLabels=False, scatter=True, t_=runningState.t)
        fig.canvas.draw()
        fig.canvas.flush_events()
        fig.savefig(os.path.join(ctx.imagePath, f'frame_{i:05d}.png'))

    if outFile is not None and (i % storeSteps == 0 or i == nSteps - 1):
        writeFrame(groups, i, stepResult.state, stepResult.stages, config=ctx.config,
                   schemeConfig=ctx.schemeConfig, uniqueParticles=True, writeStages=False,
                   extraFields=sod3dCase.extraFields)

In [ ]:
if outFile is not None:
    outFile.close()

if spec.plot:
    encodeFrames(ctx.imagePath, ctx.exportPath)

## A slice through the slab

The profile panels collapse the cross-section away on purpose. This cuts a
thin slice at $z \approx 0$ instead and draws the density in it, with the
particle visualizer the 2D examples use (`warpSPHPlotting.visualize`, through
the runner's backend-with-fallback helper). The coarser light-side lattice is
visible directly, and so is the flatness of the fronts -- a bowed contact or a
ragged shock here is the same error the profile scatter reports as vertical
spread, but located.

Slicing rather than projecting matters in 3D: drawing every particle would
stack the whole depth of the slab on top of itself, and a coarse lattice would
be indistinguishable from a genuinely disordered front. `zSlice` sets the
half-thickness -- keep it under one light-side spacing to stay a slice rather
than a projection. `xWindow` selects the piece of the tube to look at; the
default is the half the analytic solution describes.

In [ ]:
from warpSPHPlotting import PlottingOptions, UniformColorMap   # noqa: E402

xWindow = (0.0, 1.0)          # try (0.4, 0.8) to zoom on the contact and the shock
zSlice = 0.03                 # half-thickness of the slice, in domain units


def windowed(state, domain, xWindow, zSlice):
    """The particles inside an x window and a thin slice about z=0, with a
    domain box to match, ready for `visualize`.

    The slice is what makes this a 3D picture rather than a projection: passing
    every particle would draw the whole depth of the slab on top of itself, so
    a coarse light-side lattice and a genuinely ragged front would look alike.
    """
    keep = ((state.positions[:, 0] >= xWindow[0]) & (state.positions[:, 0] <= xWindow[1])
            & (state.positions[:, 2].abs() <= zSlice))
    perParticle = {f.name: getattr(state, f.name)[keep]
                   for f in dataclasses.fields(state)
                   if torch.is_tensor(getattr(state, f.name))
                   and getattr(state, f.name).shape[:1] == state.positions.shape[:1]}
    low, high = domain.min.clone(), domain.max.clone()
    low[0], high[0] = xWindow
    return (dataclasses.replace(state, **perParticle),
            type(domain)(low, high, domain.periodic, domain.dim))


slabState, slabDomain = windowed(runningState.state, ctx.config.domain, xWindow, zSlice)

plotter = visualizeWithFallback(
    ctx, 'matplotlib',        # a few thousand particles; vispy is for 10^5
    particleState=slabState,
    domain=slabDomain,
    quantities={'A': slabState.densities},
    # A uniform map, not the diverging one the 2D examples reach for: density
    # here runs monotonically from the light state to the dense one and has no
    # meaningful midpoint to diverge about.
    plotOptions={'A': PlottingOptions(
        colorMap=UniformColorMap.viridis, markerSize=6,
        plotTitle='density', vMin=0.2, vMax=1.05)},
    figTitle=f'{sod3dCase.name}  t = {float(runningState.t):.4g}  '
             f'(|z| < {zSlice:g} slice: {slabState.positions.shape[0]} of '
             f'{runningState.state.positions.shape[0]} particles)',
    mosaic='A', figsize=(11, 3),
)
plotter.export(os.path.join(ctx.imagePath, 'slab.png'), dpi=150)
figureOf(plotter)